In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm

BEST CONFIG:

BATCH SIZE = 32 
LR = 0.001
EPOCHS = 20 
ADAM 
MODEL IS NORMAL 2 LAYERS FOR BOTH 


In [6]:
# ===============================================================
# 1) CONFIGURATION
# ===============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW, INPUT_LEN, HORIZON, STRIDE = 70, 60, 10, 70
BATCH_SIZE = 32
EPOCHS = 20
LR = 0.0005

RAW_PATH = "data/New folder/train.pkl"

In [3]:
# ===============================================================
# 2) DATA PREPARATION
# ===============================================================
print("\n[1] Loading and preparing dataset...")

df = pd.read_pickle(RAW_PATH)

# Split by series_id
unique_series = sorted(df['series_id'].unique())
n_train = int(0.8 * len(unique_series))
train_ids, val_ids = unique_series[:n_train], unique_series[n_train:]

train_df = df[df['series_id'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['series_id'].isin(val_ids)].reset_index(drop=True)

print(f"   → Train series: {len(train_ids)}, Val series: {len(val_ids)}")
print(f"   → Train rows: {len(train_df):,}, Val rows: {len(val_df):,}\n")



[1] Loading and preparing dataset...
   → Train series: 40, Val series: 10
   → Train rows: 14,395,781, Val rows: 3,935,443



PRACTICE
---------

In [4]:
train_df

,series_id,time_step,close,volume
0,1,0,0.13700,171985.703125
1,1,1,0.13656,85451.398438
2,1,2,0.13647,121151.898438
3,1,3,0.13693,249110.593750
4,1,4,0.13715,280344.500000
...,...,...,...,...
14395776,40,393653,0.07469,147.699997
14395777,40,393654,0.07472,135.199997
14395778,40,393655,0.07478,123605.898438
14395779,40,393656,0.07480,54159.101562


In [5]:
trial_features_df = train_df[train_df['series_id'] == 1].copy()
trial_features_df

,series_id,time_step,close,volume
0,1,0,0.13700,171985.703125
1,1,1,0.13656,85451.398438
2,1,2,0.13647,121151.898438
3,1,3,0.13693,249110.593750
4,1,4,0.13715,280344.500000
...,...,...,...,...
394555,1,394555,0.07720,61680.000000
394556,1,394556,0.07727,12530.000000
394557,1,394557,0.07718,32552.800781
394558,1,394558,0.07723,9335.400391


In [6]:

EPS = 1e-8
WINDOW = 70
INPUT_LEN = 60
HORIZON = 10

# --- returns once, for entire series ---
trial_features_df["returns"] = trial_features_df["close"].pct_change().fillna(0)*100

trial_features_df

,series_id,time_step,close,volume,returns
0,1,0,0.13700,171985.703125,0.000000
1,1,1,0.13656,85451.398438,-0.321168
2,1,2,0.13647,121151.898438,-0.065899
3,1,3,0.13693,249110.593750,0.337064
4,1,4,0.13715,280344.500000,0.160670
...,...,...,...,...,...
394555,1,394555,0.07720,61680.000000,0.038874
394556,1,394556,0.07727,12530.000000,0.090671
394557,1,394557,0.07718,32552.800781,-0.116479
394558,1,394558,0.07723,9335.400391,0.064790


In [7]:
windows = []
for start in range(0, len(trial_features_df) - WINDOW + 1, WINDOW):
    w = trial_features_df.iloc[start:start + WINDOW].copy()
    c = w["close"].values
    v = w["volume"].values

    c_past, c_future = c[:INPUT_LEN], c[INPUT_LEN:]
    v_past = v[:INPUT_LEN]

    # --- 1. Min–max on past 60 close (for input shape) ---
    c_min, c_max = c_past.min(), c_past.max()
    c_scaled = 2 * ((c_past - c_min) / (c_max - c_min + EPS)) - 1

    # --- 2. Log1p volume + min–max ---
    v_log = np.log1p(v_past)
    v_min, v_max = v_log.min(), v_log.max()
    v_scaled = 2 * ((v_log - v_min) / (v_max - v_min + EPS)) - 1

    # --- 3. Past returns for same 60 steps ---
    r = w["returns"].values[:INPUT_LEN]

    # --- 4. TARGET = next-10 returns (each relative to its previous step) ---
    y = ((c[INPUT_LEN:INPUT_LEN + HORIZON] /
      c[INPUT_LEN - 1:INPUT_LEN - 1 + HORIZON]) - 1.0) * 100 

    # --- 5. Combine into dataframe ---
    X = pd.DataFrame({
        "time_step": np.arange(INPUT_LEN),
        "close_scaled": c_scaled,
        "volume_scaled": v_scaled,
        "returns": r
    })

    windows.append({
        "window_id": len(windows),
        "X": X,
        "y_returns": y.astype(np.float32),
        "last_close": c_past[-1],
        "c_min": c_min,
        "c_max": c_max
    })

# --- preview one window ---
w0 = windows[0]
print("INPUT (first window)")
print(w0["X"].head(10))
print("\nTARGET (next 10 returns):")
print(w0["y_returns"])


INPUT (first window)
   time_step  close_scaled  volume_scaled   returns
0          0      0.757563       0.142596  0.000000
1          1      0.461267      -0.107886 -0.321168
2          2      0.400670       0.017127 -0.065899
3          3      0.710431       0.275269  0.337064
4          4      0.858579       0.317570  0.160670
5          5      0.892244       0.331791  0.036454
6          6      0.966318      -0.023925  0.080180
7          7      0.999993       0.091467  0.036418
8          8      0.744096      -0.464049 -0.276655
9          9      0.434335       0.134369 -0.335813

TARGET (next 10 returns):
[ 0.03687143 -0.32444    -0.00739694  0.12577772  0.04432201  0.21418333
  0.16950369 -0.03679395 -0.06623268 -0.23567677]


In [9]:
trial_features_df[(trial_features_df['series_id'] == 1) & (trial_features_df['time_step'] < 11)]

,series_id,time_step,close,volume,returns
0,1,0,0.13700,171985.703125,0.000000
1,1,1,0.13656,85451.398438,-0.321168
2,1,2,0.13647,121151.898438,-0.065899
3,1,3,0.13693,249110.593750,0.337064
4,1,4,0.13715,280344.500000,0.160670
5,1,5,0.13720,291702.093750,0.036454
6,1,6,0.13731,108030.000000,0.080180
7,1,7,0.13736,149102.906250,0.036418
8,1,8,0.13698,31606.500000,-0.276655
9,1,9,0.13652,168079.593750,-0.335813


RESUME HERE
------------

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset

class CryptoWindowDataset(Dataset):
    """
    Each sample:
        X: (input_len, 3)  -> [close_scaled, volume_scaled, local_returns]
        y: (horizon,)       -> next 10 local returns (in %)
    close & volume normalized per-window to [-1, 1].
    Returns computed locally within each window (no cross-window leakage).
    """

    def __init__(self, df, window=70, input_len=60, horizon=10, stride=70):
        EPS = 1e-8
        self.X, self.y, self.c_min, self.c_max = [], [], [], []

        # ensure sorted order
        df = df.sort_values(["series_id", "time_step"]).reset_index(drop=True)
        df["volume"] = np.log1p(np.clip(df["volume"], 0, None))

        for sid, g in df.groupby("series_id"):
            c = g["close"].values.astype(np.float32)
            v = g["volume"].values.astype(np.float32)
            n = len(c)
            if n < window:
                continue

            # slide windows
            for start in range(0, n - window + 1, stride):
                w_close = c[start:start + window]
                w_vol   = v[start:start + window]

                # --- split past & future ---
                c_past, c_future = w_close[:input_len], w_close[input_len:]
                v_past = w_vol[:input_len]

                # --- per-window scaling [-1,1] ---
                c_min, c_max = c_past.min(), c_past.max()
                c_scaled = 2 * ((c_past - c_min) / (c_max - c_min + EPS)) - 1

                v_min, v_max = v_past.min(), v_past.max()
                v_scaled = 2 * ((v_past - v_min) / (v_max - v_min + EPS)) - 1

                # --- compute *local returns* within this window ---
                # percent change relative to previous close
                r_local = np.zeros_like(c_past)
                r_local[1:] = (c_past[1:] / c_past[:-1] - 1.0) * 100

                # --- inputs ---
                x = np.stack([c_scaled, v_scaled, r_local], axis=1)  # (60, 3)

                # --- targets: next-10 returns relative to last input close ---
                y = (c_future / c_past[-1] - 1.0) * 100

                self.X.append(x)
                self.y.append(y.astype(np.float32))
                self.c_min.append(c_min)
                self.c_max.append(c_max)

        # convert to arrays for efficiency
        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.float32)
        self.c_min = np.array(self.c_min, dtype=np.float32)
        self.c_max = np.array(self.c_max, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),   # (60, 3)
            torch.from_numpy(self.y[idx]),   # (10,) local returns (%)
            self.c_min[idx],
            self.c_max[idx]
        )


In [5]:
# ===============================================================
# 5) DATALOADERS
# ===============================================================
train_ds = CryptoWindowDataset(train_df)
val_ds   = CryptoWindowDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"[2] Train windows: {len(train_ds):,} | Val windows: {len(val_ds):,}\n")

[2] Train windows: 205,629 | Val windows: 56,215



In [21]:
train_ds

BIDIRECTIONAL LSTM:


In [7]:
# ===============================================================
# 3) MODEL: CNN(128x3) → LSTM(200x3) → Dense(10)
# ===============================================================
class CNNLSTM(nn.Module):
    """CNN-LSTM hybrid forecaster"""
    def __init__(self, forecast_len=HORIZON):
        super().__init__()
        # --- CNN layers ---
        self.conv1 = nn.Conv1d(3, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.drop_cnn = nn.Dropout(0.15)

        # --- LSTM layers ---
        self.lstm1 = nn.LSTM(input_size=128, hidden_size=200, num_layers=1, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(input_size=400, hidden_size=200, num_layers=1, batch_first=True, bidirectional=True)
        self.drop_rnn = nn.Dropout(0.15)

        # --- Output head ---
        self.fc = nn.Linear(400, forecast_len)

    def forward(self, x):
        # x: (B, 60, 3)
        x = x.transpose(1, 2)        # (B, 3, 60)
        x = F.relu(self.conv1(x)); x = self.drop_cnn(x)
        x = F.relu(self.conv2(x)); x = self.drop_cnn(x)
        x = x.transpose(1, 2)        # (B, 60, 128)

        x, _ = self.lstm1(x); x = self.drop_rnn(x)
        x, _ = self.lstm2(x); x = self.drop_rnn(x)

        last = x[:, -1, :]           # (B, 400)
        return self.fc(last)         # (B, 10)


In [9]:
# ===============================================================
# 6) MODEL / LOSS / OPTIMIZER
# ===============================================================
model = CNNLSTM().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.MSELoss()
print(f"[3] Model ready on {DEVICE}\n")

[3] Model ready on cuda



In [10]:
# ===============================================================
# 7) TRAINING LOOP (stateless, batch-wise, with early stopping)
# ===============================================================
print("[4] Starting training with Early Stopping...\n")

best_val_loss = float("inf")
patience, patience_counter = 5, 0  # stop if no improvement for 5 epochs

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, total = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", ncols=100)
    for step, (xb, yb, cmin, cmax) in enumerate(pbar, 1):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        total += xb.size(0)

        if step % 100 == 0 or step == len(pbar):
            pbar.set_postfix({"train_loss": f"{loss.item():.6f}"})

    train_loss /= total

    # ---------------- Validation ----------------
    model.eval()
    val_loss, total_val = 0.0, 0
    with torch.no_grad():
        pbar_val = tqdm(val_loader, desc=f"[Val] Epoch {epoch}", ncols=100)
        for step, (xb, yb, cmin, cmax) in enumerate(pbar_val, 1):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)

            val_loss += loss.item() * xb.size(0)
            total_val += xb.size(0)

            if step % 100 == 0 or step == len(pbar_val):
                pbar_val.set_postfix({"val_loss": f"{loss.item():.6f}"})

    val_loss /= total_val
    print(f"Epoch {epoch:02d} → Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

    # ---------------- Early Stopping ----------------
    if val_loss < best_val_loss - 1e-6:  # small delta to ignore noise
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"   ✅ Improvement detected. Model saved (val_loss={val_loss:.6f}).\n")
    else:
        patience_counter += 1
        print(f"   ⚠️  No improvement for {patience_counter}/{patience} epochs.\n")

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}. Best Val MSE: {best_val_loss:.6f}")
            break


[4] Starting training with Early Stopping...



[Val] Epoch 1: 100%|█████████████████████████| 1757/1757 [01:23<00:00, 20.94it/s, val_loss=0.034995]


Epoch 01 → Train MSE: 0.202939 | Val MSE: 0.129676
   ✅ Improvement detected. Model saved (val_loss=0.129676).



[Val] Epoch 2: 100%|█████████████████████████| 1757/1757 [01:37<00:00, 18.10it/s, val_loss=0.034149]


Epoch 02 → Train MSE: 0.202008 | Val MSE: 0.129756
   ⚠️  No improvement for 1/5 epochs.



...

Epoch 05 → Train MSE: 0.201024 | Val MSE: 0.128259
   ⚠️  No improvement for 1/5 epochs.



[Val] Epoch 6: 100%|█████████████████████████| 1757/1757 [01:51<00:00, 15.74it/s, val_loss=0.035963]


Epoch 06 → Train MSE: 0.200508 | Val MSE: 0.128482
   ⚠️  No improvement for 2/5 epochs.



Epoch 7/20:  64%|████████████████▋         | 4118/6426 [12:00<06:43,  5.72it/s, train_loss=0.432463]


KeyboardInterrupt: 

In [18]:
# ===============================================================
# 8) SAVE MODEL
# ===============================================================
torch.save(model.state_dict(), "model_weights_new_feature.pkl")
print("\n[5] Training complete — saved weights to model_weights_new_feature.pkl")


[5] Training complete — saved weights to model_weights_new_feature.pkl
